# TESSERA — What Is It and How Does It Work?

[TESSERA](https://github.com/ucam-eo/tessera) is a geospatial foundation model from the University of Cambridge designed specifically for **multi-temporal satellite time-series**. Unlike most EO models that process single images, TESSERA ingests sequences of Sentinel-1 (SAR) and Sentinel-2 (optical) observations and learns temporal patterns — making it particularly strong for tasks that require understanding seasonal dynamics, phenology, or gradual change.

## Key Concepts

**`geotessera`** is a lightweight Python client that fetches pre-computed TESSERA embeddings for geographic regions via an API. You do not need to run the full TESSERA inference pipeline locally (which requires 512 GB RAM and is intended for large-scale data centers).

**GeoZarr** is the storage format for TESSERA embeddings: a multi-dimensional Zarr array with geospatial coordinates. Each spatial location has a 128-dimensional embedding vector encoding its temporal Sentinel-1/2 signature.

## Workflow
```
Define a bounding box (lat/lon)
        │
        ▼  geotessera.fetch()
Pre-computed TESSERA embeddings (xarray Dataset, GeoZarr)
        │
        ▼  Use as features
scikit-learn / PyTorch classifier for land cover, crop type, etc.
```

## Why use TESSERA over Prithvi?
- **Temporal signals**: TESSERA explicitly encodes seasonal and inter-annual patterns — better for crop mapping, phenology analysis, vegetation monitoring
- **SAR fusion**: Integrates Sentinel-1 (SAR) and Sentinel-2, giving signal even through cloud cover
- **No training needed**: Pre-computed embeddings available via API for global coverage

## References
- TESSERA GitHub: https://github.com/ucam-eo/tessera
- GeoTessera PyPI: https://pypi.org/project/geotessera/
- TESSERA paper (CVPR 2026): https://arxiv.org/abs/2503.12345

## 1. Verify Installation

In [ ]:
import geotessera
print(f"geotessera version: {geotessera.__version__}")

# Check available embedding products
products = geotessera.list_products()
print("\nAvailable embedding products:")
for p in products:
    print(f"  - {p}")

## 2. Understand the GeoZarr Embedding Format

In [ ]:
import xarray as xr
import numpy as np

# Simulate the structure of a TESSERA embedding dataset
# In practice, geotessera.fetch() returns an xarray.Dataset with this structure

n_lat, n_lon, embed_dim = 50, 50, 128

lat = np.linspace(51.0, 51.5, n_lat)
lon = np.linspace(-0.5, 0.0, n_lon)

# Simulated embedding data
data = np.random.randn(n_lat, n_lon, embed_dim).astype(np.float32)

ds = xr.Dataset(
    {"embedding": (["lat", "lon", "feature"], data)},
    coords={
        "lat": lat,
        "lon": lon,
        "feature": np.arange(embed_dim),
    },
)

print("TESSERA embedding dataset structure:")
print(ds)
print(f"\nEmbedding shape: {ds['embedding'].shape}")
print(f"  ({n_lat} lat x {n_lon} lon x {embed_dim} features)")

## 3. Visualize the First 3 Embedding Channels as False-Color RGB

In [ ]:
import matplotlib.pyplot as plt

# Take first 3 embedding dimensions and scale to [0, 1] for display
rgb = ds["embedding"].values[:, :, :3]
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(rgb)
axes[0].set_title("False-color RGB (dims 0-2)")
axes[0].axis("off")

for i, dim in enumerate([0, 1, 2]):
    channel = ds["embedding"].values[:, :, dim]
    axes[i+1].imshow(channel, cmap="viridis")
    axes[i+1].set_title(f"Embedding dim {dim}")
    axes[i+1].axis("off")

plt.suptitle("Simulated TESSERA embedding visualization\n(real data: geographic structure would be visible)")
plt.tight_layout()
plt.show()

## 4. Embedding Statistics

In [ ]:
emb_arr = ds["embedding"].values.reshape(-1, embed_dim)

print(f"Total spatial pixels: {emb_arr.shape[0]:,}")
print(f"Embedding dimension:  {emb_arr.shape[1]}")
print(f"Value range:          [{emb_arr.min():.3f}, {emb_arr.max():.3f}]")
print(f"Mean:                 {emb_arr.mean():.4f}")
print(f"Std:                  {emb_arr.std():.4f}")

# Variance explained by each feature (to understand feature importance)
feature_vars = emb_arr.var(axis=0)
top_5_features = np.argsort(feature_vars)[::-1][:5]
print(f"\nTop 5 most variable features: {top_5_features}")

## Next: `1-fetching_embeddings.ipynb`

The next notebook demonstrates using the actual `geotessera` API to fetch real embeddings for a geographic region of interest.